# Μῆτις (Metis) — Train on Google Colab GPU

Train your Metis model on Colab's free GPU and save the checkpoints to your **Google Drive**.

**8 simple steps — just run each cell top to bottom:**
1. Mount Google Drive
2. Clone your private GitHub repo (`iamasrakib/Metis`) with a Personal Access Token
3. Install dependencies
4. Generate the West Bengal dataset
5. Link the checkpoint folder into Google Drive (checkpoints save there as it trains)
6. Train on the GPU — `train_westbengal_100m.py` (100M params)
7. Confirm the checkpoints landed in Drive
8. (Optional) Generate a sample from the trained model

> **One-time token setup:** GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token → tick **repo** → copy it. You'll paste it into Step 2.

---
## Step 1 — Mount Google Drive

Your checkpoints will be stored here: `MyDrive/Metis/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Clone your private Metis repo

Run the next cell, then **paste your GitHub token** into the box that appears.

In [ ]:
import getpass, os

GITHUB_TOKEN = getpass.getpass("Paste GitHub Personal Access Token (repo scope): ")
REPO = f"https://{GITHUB_TOKEN}@github.com/iamasrakib/Metis.git"
METIS_DIR = "/content/Metis"

if os.path.isdir(METIS_DIR):
    ret = os.system(f"git -C {METIS_DIR} pull --ff-only")
else:
    ret = os.system(f"git clone {REPO} {METIS_DIR}")
if ret != 0:
    raise RuntimeError("git failed - check the token has 'repo' scope and try again.")

%cd /content/Metis

---
## Step 3 — Install dependencies

PyTorch (with CUDA) is already installed on Colab — this installs the small set of extra packages Metis needs for training.

In [ ]:
!pip install -q numpy tqdm tiktoken tokenizers

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("No GPU found! Enable one: Runtime > Change runtime type > T4 GPU")

---
## Step 4 — Generate the dataset

The generator scripts are self-contained (the text is baked in), so nothing to download. Default: **West Bengal**. Switch by uncommenting another line.

In [ ]:
!python data/generate_westbengal_dataset.py
# !python data/generate_cow_dataset.py      # cow knowledge corpus
# !python data/generate_siraj_dataset.py    # Siraj-ud-Daulah history

---
## Step 5 — Link checkpoints straight into Google Drive

Creates `MyDrive/Metis/checkpoints_westbengal_100m/` and symlinks it to where training writes. Every checkpoint is **saved directly to Drive as training runs** — even if this Colab session disconnects mid-training, nothing is lost.

In [ ]:
import os

CKPT_DIR   = "checkpoints_westbengal_100m"       # matches train_westbengal_100m.py
DRIVE_BASE = "/content/drive/MyDrive/Metis"

os.makedirs(DRIVE_BASE, exist_ok=True)
drive_ckpt = os.path.join(DRIVE_BASE, CKPT_DIR)
os.makedirs(drive_ckpt, exist_ok=True)

local_link = os.path.abspath(CKPT_DIR)
if not os.path.lexists(local_link):
    os.symlink(drive_ckpt, local_link)
    print(f"Linked  {local_link}\n     -> {drive_ckpt}")
else:
    print("Already linked:", local_link)

print("\nTrain a different model? Change CKPT_DIR to match the script:")
print("  train_westbengal_100m.py  -> checkpoints_westbengal_100m")
print("  train_cow.py              -> checkpoints_cow")
print("  train_westbengal_small.py -> checkpoints_westbengal_small")

---
## Step 6 — Train on the GPU

The 100M config ships with `max_iters=30` (just a smoke test). Set how many steps you want — for this 36K-char corpus **500–2000** works well. Training prints live progress.

In [ ]:
import re

MAX_ITERS = 1000   # <-- how many optimizer steps to train

src = "train_westbengal_100m.py"
with open(src, encoding="utf-8") as f:
    code = f.read()
code = re.sub(r"max_iters=\d+", f"max_iters={MAX_ITERS}", code)
with open(src, "w", encoding="utf-8") as f:
    f.write(code)
print(f"max_iters -> {MAX_ITERS}")

# Train on the Colab GPU. Checkpoints stream into Google Drive (Step 5).
!python train_westbengal_100m.py

---
## Step 7 — Confirm the checkpoints are in Drive

In [ ]:
import os

ckpt_dir = "/content/drive/MyDrive/Metis/checkpoints_westbengal_100m"
for name in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, name)
    size = os.path.getsize(path)
    line = f"  {name:<28} {size/1e6:6.2f} MB" if size >= 1e6 else f"  {name:<28} {size:>7,} B"
    print(line)

print("\nSaved in Google Drive: MyDrive/Metis/checkpoints_westbengal_100m/")
print("Chat with it on your PC later:  metis chat --checkpoint-dir checkpoints_westbengal_100m")

---
## Step 8 — (Optional) Test the model here

Generates a short continuation on the GPU from the freshly trained weights.

In [ ]:
!python generate.py --prompt "West Bengal is" --max-tokens 150 --checkpoint-dir checkpoints_westbengal_100m